# Decision Tree Classification of Breast Cancer Types from Gene Expression Data
This notebook trains a single decision tree on the `ML_HW_Data_CancerGene.xlsx` dataset to classify breast cancer tumor types and evaluates per-class precision, recall, and F-measure.

## 1. Import Required Libraries

In [24]:
import pandas as pd
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

## 2. Load Data from Excel Sheets

In [25]:
file_path = 'ML_HW_Data_CancerGene.xlsx'

# Use header=None because the first row is data, not column names
X_train = pd.read_excel(file_path, sheet_name='geneexpTrain', header=None)
Y_train = pd.read_excel(file_path, sheet_name='tumortypeTrain', header=None)
X_test = pd.read_excel(file_path, sheet_name='geneexpTest', header=None)
Y_test = pd.read_excel(file_path, sheet_name='tumortypeTest', header=None)

print("Data loaded successfully!")
print(f"X_train shape: {X_train.shape}")
print(f"Y_train shape: {Y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"Y_test shape:  {Y_test.shape}")

Data loaded successfully!
X_train shape: (63, 2308)
Y_train shape: (63, 1)
X_test shape:  (25, 2308)
Y_test shape:  (25, 1)


## 3. Explore Training and Testing Data

In [26]:
# Flatten Y to 1-D arrays
y_train = Y_train.iloc[:, 0]
y_test = Y_test.iloc[:, 0]

# The 5th class in test set appears as NaN — label it as "Unknown"
y_test = y_test.fillna("Unknown")

print("=== Training Labels ===")
print(f"Unique classes ({y_train.nunique()}): {sorted(y_train.unique())}")
print(y_train.value_counts().sort_index())

print("\n=== Testing Labels ===")
print(f"Unique classes ({y_test.nunique()}): {sorted(y_test.unique())}")
print(y_test.value_counts().sort_index())

print(f"\n*** Key finding: Training has {y_train.nunique()} classes, "
      f"Testing has {y_test.nunique()} classes ***")
print("*** The 'Unknown' class exists ONLY in the test set — the model was never trained on it! ***")

=== Training Labels ===
Unique classes (4): ['BL', 'EW', 'NB', 'RM']
0
BL     8
EW    23
NB    12
RM    20
Name: count, dtype: int64

=== Testing Labels ===
Unique classes (5): ['BL', 'EW', 'NB', 'RM', 'Unknown']
0
BL         3
EW         6
NB         6
RM         5
Unknown    5
Name: count, dtype: int64

*** Key finding: Training has 4 classes, Testing has 5 classes ***
*** The 'Unknown' class exists ONLY in the test set — the model was never trained on it! ***


## 4. Train a Decision Tree Classifier

In [27]:
dt_clf = DecisionTreeClassifier(random_state=42)
dt_clf.fit(X_train, y_train)
print(f"Decision Tree trained with max_depth = {dt_clf.get_depth()}, "
      f"number of leaves = {dt_clf.get_n_leaves()}")

Decision Tree trained with max_depth = 4, number of leaves = 6


## 5. Predict on Test Set

In [28]:
y_pred = dt_clf.predict(X_test)

results_df = pd.DataFrame({'Actual': y_test, 'Predicted': y_pred})
print(results_df.to_string())
print(f"\nOverall Accuracy: {(y_pred == y_test).mean():.4f}")

     Actual Predicted
0        BL        EW
1        BL        EW
2        BL        RM
3        EW        NB
4        EW        RM
5        EW        EW
6        EW        EW
7        EW        NB
8        EW        RM
9   Unknown        EW
10  Unknown        BL
11  Unknown        EW
12  Unknown        RM
13  Unknown        NB
14       NB        NB
15       NB        NB
16       NB        RM
17       NB        BL
18       NB        EW
19       NB        RM
20       RM        EW
21       RM        RM
22       RM        RM
23       RM        RM
24       RM        NB

Overall Accuracy: 0.2800


## 6. Per-Class Precision, Recall, and F-Measure

In [29]:
# Get all unique labels from both train and test
all_labels = sorted(set(y_train.unique()) | set(y_test.unique()))

print("Classification Report (per-class Precision / Recall / F1-Score):\n")
report = classification_report(y_test, y_pred, labels=all_labels, zero_division=0)
print(report)

print("\nConfusion Matrix:")
cm = confusion_matrix(y_test, y_pred, labels=all_labels)
cm_df = pd.DataFrame(cm, index=[f"Actual: {l}" for l in all_labels],
                         columns=[f"Pred: {l}" for l in all_labels])
print(cm_df)

Classification Report (per-class Precision / Recall / F1-Score):

              precision    recall  f1-score   support

          BL       0.00      0.00      0.00         3
          EW       0.25      0.33      0.29         6
          NB       0.33      0.33      0.33         6
          RM       0.33      0.60      0.43         5
     Unknown       0.00      0.00      0.00         5

    accuracy                           0.28        25
   macro avg       0.18      0.25      0.21        25
weighted avg       0.21      0.28      0.23        25


Confusion Matrix:
                 Pred: BL  Pred: EW  Pred: NB  Pred: RM  Pred: Unknown
Actual: BL              0         2         0         1              0
Actual: EW              0         2         2         2              0
Actual: NB              1         1         2         2              0
Actual: RM              0         1         1         3              0
Actual: Unknown         1         2         1         1              0


## 7. Analysis and Special Observations

**Key Observation:** The testing set contains **5 classes** while the training set has only **4 classes**. This means there is at least one cancer type in the test data that the decision tree **never saw during training**.

**Consequences:**
- The model **cannot predict** the unseen class, so any test samples belonging to that class will always be misclassified.
- That class will have **0.00 recall** (the model never predicts it correctly) and **0.00 precision** (if the model never outputs that label) — resulting in an **F1-score of 0.00**.
- This artificially lowers the overall accuracy and macro-average metrics.

**Real-world implication:** This is a common problem in classification — if a new class appears at inference time that was absent from training, the model has no way to identify it. Solutions include:
1. Ensuring the training set is representative of all possible classes.
2. Using novelty/outlier detection to flag samples that don't match known classes.
3. Retraining or updating the model when new classes are discovered.